# Simple Agentic RAG

This notebook walks through building an Agentic RAG system using LangChain and FAISS.

**Steps:**
1. Install dependencies
2. Ingest documents into a FAISS vector store
3. Build a LangChain agent with a document search tool
4. Ask questions

## 1. Install Dependencies

In [1]:
%pip install -r  requirements.txt -q


Note: you may need to restart the kernel to use updated packages.


## 2. Set API Keys

In [2]:
import os

from dotenv import load_dotenv


#os.environ["HUGGINGFACEHUB_API_TOKEN"] = HF_TOKEN
load_dotenv(os.path.expanduser("~/FDP-AGENENTIC-AI-RAG/.env"))
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
HF_TOKEN = os.getenv


# huggingface_hub caches HF_HUB_OFFLINE at import time — reset it directly
#import huggingface_hub.constants as _hf_const
#_hf_const.HF_HUB_OFFLINE = False

## 3. Ingest Documents into FAISS Vector Store

Defines knowledge inline, splits it into chunks, embeds them, and saves the vector store locally.

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

fictitious_department_info = [
    "The Department of Intelligent Systems Engineering (DISE) is a small, focused department that works on applied AI and intelligent systems.",
    "It currently has around 40 students, with a healthy mix of undergraduate and postgraduate learners.",
    "The department is run by a team of 14 professors, including experienced faculty members and a few industry practitioners.",
    "Students can choose from about 5 courses, ranging from core subjects to electives and hands-on project work.",
    "Overall, DISE aims to prepare students for real-world engineering roles through practical learning and industry exposure.",
]

documents = [Document(page_content=text) for text in fictitious_department_info]

splitter = RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=50)
docs = splitter.split_documents(documents)

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L12-v2",
    model_kwargs={"local_files_only": True},
)
db = FAISS.from_documents(docs, embeddings)
db.save_local("vectorstore")

print(f"Vector DB created successfully ({len(docs)} chunks)")

/tmp/ipykernel_48700/2126607911.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Vector DB created successfully (5 chunks)


## 4. Build the RAG Agent

Loads the vector store and wires it up as a tool for a LangChain ZERO_SHOT_REACT agent.

In [ ]:
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.chat_models import init_chat_model

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L12-v2",
    model_kwargs={"local_files_only": True},
)

db = FAISS.load_local(
    "vectorstore",
    embeddings,
    allow_dangerous_deserialization=True
)

@tool
def search_answers(query: str) -> str:
    """Search the knowledge base"""
    results = db.similarity_search(query, k=2)
    return "\n".join([doc.page_content for doc in results])

# Switch providers via env vars (set before running this cell):
#   google_genai → MODEL_NAME=google_genai:gemini-2.5-flash  (default)
#   openai       → MODEL_NAME=openai:gpt-4o
#   anthropic    → MODEL_NAME=anthropic:claude-3-5-sonnet-20241022
#   ollama       → MODEL_NAME=ollama:llama3
#   huggingface  → MODEL_NAME=huggingface:Qwen/Qwen2.5-7B-Instruct
MODEL_NAME = os.getenv("MODEL_NAME", "google_genai:gemini-2.5-flash")

chat_llm = init_chat_model(
    MODEL_NAME,
    google_api_key=GEMINI_API_KEY,
    temperature=0.1,
)

agent = create_agent(
    model=chat_llm,
    tools=[search_answers],
    system_prompt=(
        "You are an intelligent Agentic RAG system. "
        "Decide whether document retrieval is needed. "
        "If needed, use the search_docs tool."
    ),
)

print(f"Agent ready (model: {MODEL_NAME}).")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Agent ready (model: google_genai:gemini-2.5-flash).


## 5. Ask Questions

Edit `question` below and re-run this cell to query the agent.

In [5]:
def ask(question: str):
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    print("\nAnswer:", result["messages"][-1].content)


ask("What is DISE?")


Answer: [{'type': 'text', 'text': 'The Department of Intelligent Systems Engineering (DISE) is a small, focused department that works on applied AI and intelligent systems. Its main goal is to prepare students for real-world engineering roles through practical learning and industry exposure.', 'extras': {'signature': 'CtEDAQw51sc/sA+IR+Voq3G9BWLKRtiXMRMHXqlo+Lqk+qX4oX6c7vEf6tR8t1VsNdl0lL9AYsILqmGUbqEzErWVIO6Kqld07RPZEiMuTuTcYRA4Zbg/vVMibm6rK66WD8JwuNF0MpuXv+1nFCSAtaM1lCfMBNrfjgoPlRaHgnRlC/glqGM13E3mtGAdphiaNz5VBY2A6Ed3fOTAJoStHIr5Og2d7DjroFZ0krXzJUYL1CwsGV7+D41vwJzJH0XXPP/aCqbiEJpxJBmvrcgMxoFECEJvVZd7XqcpesTz8bEwx0V4mvVrFfOz4PthVkmCiGgwxgHAhNXUsynbtAP67f1di3qGAfeIpdKLQEZn07mzEX+b9m6Jc0n+6Est6W6VdGLA37CsIl2XUJlip1bhi1vo6aOhFNmJWho458kUynhIq59/lChVXtTJt03JLIYVwIXIBZmIVirpX+G8c5KmxzBg0o4wpUwSvM5cQiv2r4TxZTaUtvtI7qGRM+AuYlY2RpdV+tbbrqdBWyFoDvZ3dePnpv26GwazzcXZRPTBapOLYJQbFKSSjwc1Y61yDWh2QCw2UMIKMmcH1kEeGgjDEWWxPPTtFJDk4aBTc2uWqzGZhrue'}}]


In [6]:
ask("How many students does DISE have?")


Answer: [{'type': 'text', 'text': 'DISE currently has around 40 students.', 'extras': {'signature': 'CqMCAQw51scN5ZYUTQZm5wBgA9gGIP10yZTBjcGC9Bu/mfhGc6r3FCEewEP8CQ8plht19kZI6sAqfA0Gqx/qhGDrIomhM4Tz+4A5p+MJNfQLK4sf/YewFilHBa63qLbfFXTlR0iaQSeyMoRypKexNRX/RQOFz0lA+ftL1gMIlKfBi//5Y1rf+nNwOs4W0/y/6tgMqbpGkjFEx3XJqPlAF8B1oE14MjaLqCo1HzM/sK+MIZfffuQniQRmKTUV8pa547qErX9Xg3hFf4JymIPpBGMufUVvim0Xb3PTg65CLIO1GUoJ+Qvb+6PEg8Mp375jJr7eEegHK9Y8t+dEAw7ETbYE2oqu8UDA44QRNSSHtI8T2IamMUWxIWgNj/YcZncPY+PeAL0Y'}}]


In [7]:
ask("How many stuents does it have")


Answer: [{'type': 'text', 'text': 'It currently has around 40 students, with a healthy mix of undergraduate and postgraduate learners.', 'extras': {'signature': 'Cr8DAQw51seLyZC/QbJNUFJ9EGPR/86K+/fr9Z3lm495Bp/IUu1UF9sKPb4zrXmO7+cJ27eakXURNbMdS6m8FGkv+sfyh1GxUcXhGelZL+bfVRQckY2UGlwCMVWMTHlZI513O4u3d0YIDheNqD7FTxwQ7EwqgNst3H4AIb6seJWhPXOfMhEFg77Mpp0xpokOIVTY2N18mEhq5NOicR2oSZTOyKiJZASjcHXKLhfCFWTIumNpDMns2iHWGzqzTb00PScB/6UKNe/BSK6Uhx95Cpqn3XGei1ZGfTp39m95dohRSKMDXeMbnDFBs3gDLu4NFW0Dudz/14StJRZBMU/g4E/+AkZGcudYcBATygbNh9Ed1WIMx1tOHKGN7IPowkCN+HB+2S90XIoisvoMSSfkEQEE6/mgUGaUUly71vAJwz9IpWQMpKJ8TLpnhhgRbwseG+lbtErAfNtdPuuFlTBMz+K7VVahcopqhKR6crqaPz/RP0M01Ma27D/246igvdCn+j458+viwEabPLkj4wG2hw2+DEOOVt0ewCUenYYLWDWAtreFHpT58hvv7riMDpHaZ0R1ko1jI2XbjmBUrDRRwyXU'}}]
